# Rank Fusion and the Geometry of Rank Aggregation — companion notebook

This notebook is the **narrative** half of the rank-fusion Python pillar. The
reusable, tested implementation lives next door in [`rank_fusion_rrf.py`](rank_fusion_rrf.py);
here we import it and walk the topic section by section, so every claim the page
makes renders as executed output.

**Two artifacts, one source of truth.** `rank_fusion_rrf.py` owns the numbers — its
`assert`-based harness encodes scale invariance, the Diaconis–Graham inequality, the
RRF→Borda limit, the footrule 2-approximation of Kemeny, and the hybrid NDCG win — and
its finance corpus and fused ranking are mirrored *to the decimal* by
`FusionLaboratory.tsx`. This notebook never redefines that math; it calls into it.

Run from the repo root:

```
uv run --with numpy --with scipy --with jupyter \
    jupyter execute notebooks/rank-fusion-rrf/01_rank_fusion_rrf.ipynb
```

or run the reference implementation directly, which executes the same checks:

```
uv run --with numpy --with scipy python notebooks/rank-fusion-rrf/rank_fusion_rrf.py
```

In [ ]:
import sys
import pathlib

# Make rank_fusion_rrf.py importable whether the kernel starts in the repo root or
# in this notebook's own directory. (It in turn imports the BM25 lexical leg.)
_here = pathlib.Path.cwd()
for _cand in (_here, _here / "notebooks" / "rank-fusion-rrf", pathlib.Path("notebooks/rank-fusion-rrf")):
    if (_cand / "rank_fusion_rrf.py").exists():
        sys.path.insert(0, str(_cand))
        break

from rank_fusion_rrf import (
    _FINANCE_CORPUS, _QUERY, _QRELS,
    lexical_scores, toy_dense_scores, order_of,
    rrf_fuse, rrf_contributions, borda, combsum,
    kendall_tau, spearman_footrule,
    footrule_aggregate, kemeny_bruteforce, kemeny_cost,
    ndcg_at_k,
    test_scale_invariance, test_diaconis_graham, test_rrf_to_borda,
    test_footrule_2approx, test_hybrid_beats_either_leg, finance_demo,
)

## 1. Two retrievers, two rankings

We retrieve over a small finance corpus — 10-K filing text and earnings-call
passages — for the query *interest rate exposure*. The **lexical** leg is BM25
(imported from the BM25 topic); the **dense** leg is a deterministic toy embedding,
a seeded random projection of bag-of-words counts scored by cosine. They live on
incompatible scales — BM25 is unbounded, cosine lies in $[-1, 1]$ — and they
disagree: each ranks a genuinely on-point disclosure first that the other buries.

In [ ]:
lex = lexical_scores(_QUERY, _FINANCE_CORPUS)
den = toy_dense_scores(_QUERY, _FINANCE_CORPUS)
lex_order, den_order = order_of(lex), order_of(den)

print(f"query = {_QUERY!r}   N = {len(_FINANCE_CORPUS)} documents\n")
print(f"{'rank':<6}{'lexical (BM25)':<22}{'score':<10}{'dense (cosine)':<22}{'score'}")
for i in range(len(lex_order)):
    a, b = lex_order[i], den_order[i]
    print(f"{i+1:<6}{a:<22}{lex[a]:<10.3f}{b:<22}{den[b]:.3f}")

## 2. Score fusion, and why it breaks

The naive fix is CombSUM: add the two scores. But adding numbers on incompatible
scales lets the wider-range list dominate. We show this directly: multiply the BM25
scores by 1000 — a change no monotone reweighting should be allowed to matter — and
CombSUM's order collapses onto the BM25 order. RRF, which reads only *positions*,
does not move at all. This scale invariance is asserted in `rank_fusion_rrf.py`.

In [ ]:
lex_x1000 = {d: s * 1000.0 for d, s in lex.items()}
print("CombSUM (raw scores):       ", combsum([lex, den]))
print("CombSUM (BM25 x1000):       ", combsum([lex_x1000, den]))
print("BM25 order alone:           ", lex_order)
print()
print("RRF (raw):                  ", rrf_fuse([lex_order, den_order]))
print("RRF (BM25 x1000 -> same ranks):", rrf_fuse([order_of(lex_x1000), den_order]))
print()
test_scale_invariance()

## 3. Reciprocal Rank Fusion

RRF scores a document by summing $1/(k + r)$ over its rank $r$ in each list, with
$k = 60$ the standard constant (Cormack et al. 2009). Because the score depends only
on rank, the two incompatible scales never meet. We read off the fused ranking and the
per-list $1/(k+r)$ contributions — the same numbers the viz shows on hover.

In [ ]:
fused = rrf_fuse([lex_order, den_order], k=60)
contrib = rrf_contributions([lex_order, den_order], k=60)
print("fused (RRF, k=60):", fused, "\n")
print(f"{'document':<18}{'lex 1/(k+r)':<14}{'dense 1/(k+r)':<16}{'RRF score'}")
for d in fused:
    cl, cd = contrib[d]
    print(f"{d:<18}{cl:<14.5f}{cd:<16.5f}{cl + cd:.6f}")

## 4. RRF is a positional voting rule: the Borda limit

Reading only positions makes RRF a *positional voting rule*. As $k \to \infty$ the
weights $1/(k+r)$ flatten toward a linear-in-rank profile, and the RRF order converges
to the **Borda count** — provided the Borda totals are strict. (Our symmetric finance
instance has a three-way Borda tie among the top documents, which RRF's curvature
breaks; the harness therefore tests the limit on strict random instances.)

In [ ]:
print("Borda order:        ", borda([lex_order, den_order]))
print("RRF at k = 1e6:     ", rrf_fuse([lex_order, den_order], k=1e6))
print("RRF at k = 60:      ", rrf_fuse([lex_order, den_order], k=60))
print()
test_rrf_to_borda()

## 5. The geometry of rankings: Kendall-tau and the footrule

Two rankings are two points in the space of permutations. The **Kendall-tau** distance
$K$ counts discordant (mis-ordered) pairs; the **Spearman footrule** $F$ sums absolute
position displacements. The Diaconis–Graham inequality (1977) ties them together:
$K \le F \le 2K$. We measure both between our two legs, then assert the inequality over
thousands of random permutation pairs.

In [ ]:
K = kendall_tau(lex_order, den_order)
F = spearman_footrule(lex_order, den_order)
print(f"between the lexical and dense legs:  K = {K} discordant pairs,  F = {F}")
print(f"Diaconis-Graham:  {K} <= {F} <= {2 * K}  ->  {K <= F <= 2 * K}")
print()
test_diaconis_graham()

## 6. The Kemeny consensus

The **Kemeny** consensus is the median permutation: the ranking minimizing the total
Kendall-tau distance to the input lists. For our six documents we can find it by exact
search. It is illuminating to compare it with RRF: `filing-hedging` is *every leg's #2*,
so the optimal consensus elevates it to #1 — yet the RRF heuristic ranks it only #3.
RRF is not the consensus; that gap is the honest caveat of the topic.

In [ ]:
kem, kem_c = kemeny_bruteforce([lex_order, den_order])
print("Kemeny consensus:", kem)
print(f"  total Kendall-tau to the two legs: {kem_c}")
print("RRF (k=60):      ", fused)
print(f"  Kemeny puts {kem[0]!r} first; RRF ranks it #{fused.index(kem[0]) + 1}")

## 7. A computable surrogate: the footrule 2-approximation

Kemeny is NP-hard in general (via minimum feedback arc set). The **footrule-optimal**
aggregate, by contrast, is computed in polynomial time as a min-cost bipartite matching
of documents to positions — and by Diaconis–Graham its Kemeny cost is within $2\times$
the optimum (Dwork et al. 2001). We check both the bound and the guarantee.

In [ ]:
lists = [lex_order, den_order]
agg = footrule_aggregate(lists)
opt, opt_c = kemeny_bruteforce(lists)
print("footrule aggregate:  ", agg, "  Kemeny cost", kemeny_cost(agg, lists))
print("Kemeny optimum:      ", opt, "  Kemeny cost", opt_c)
print(f"within 2x?  {kemeny_cost(agg, lists)} <= {2 * opt_c}  ->  {kemeny_cost(agg, lists) <= 2 * opt_c}")
print()
test_footrule_2approx()

## 8. Does fusion actually help? The hybrid NDCG win

Finally, relevance. Against the judgments `_QRELS`, each leg buries one prize
disclosure at #3 and scores NDCG 0.98; RRF interleaves the two leaders and recovers
the ideal ordering, NDCG 1.0 — strictly beating both. That is the empirical payoff of
rank fusion, and the closing `assert` of the harness.

In [ ]:
print(f"NDCG@10  lexical = {ndcg_at_k(lex_order, _QRELS):.3f}")
print(f"NDCG@10  dense   = {ndcg_at_k(den_order, _QRELS):.3f}")
print(f"NDCG@10  RRF     = {ndcg_at_k(fused, _QRELS):.3f}")
print()
test_hybrid_beats_either_leg()
print()
finance_demo()

---

Every numerical claim above is also an `assert` in
[`rank_fusion_rrf.py`](rank_fusion_rrf.py), so running that file as a script is a
regression test for the whole topic:

```
uv run --with numpy --with scipy python notebooks/rank-fusion-rrf/rank_fusion_rrf.py
```

and `FusionLaboratory.tsx` reproduces this exact corpus, the two input rankings, and
the $k = 60$ fused order. The three pillars — math, viz, and code — agree by
construction.